<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day1_PubMed_Fetch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.0 MB/s eta 0:00:00


In [3]:
!pip install biopython
import json
from Bio import Entrez

# 1. NCBI requires an email address to track usage
Entrez.email = "sesimboyle@gmail.com" # <-- REPLACE THIS WITH YOUR REAL EMAIL

# 2. Define the search query (tailored to our target roles)
query = "(single-cell OR spatial transcriptomics) AND (human brain) AND (novel cell type)"
max_records = 10 # Starting with 10 to test. We can increase to 50 later.

print(f"Searching PubMed for: {query}\n")

# 3. Search for PubMed IDs (PMIDs)
search_handle = Entrez.esearch(db="pubmed", term=query, retmax=max_records, sort="relevance")
search_results = Entrez.read(search_handle)
search_handle.close()

pmids = search_results["IdList"]
print(f"Found {len(pmids)} records. PMIDs: {pmids}\n")

# 4. Fetch the actual abstracts for these PMIDs
if pmids:
    fetch_handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="xml")
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    # 5. Extract and structure the data we care about
    extracted_data = []

    # Entrez.read for efetch with retmode='xml' typically returns a dictionary
    # where the top-level key is 'PubmedArticleSet', containing a list or dictionary of 'PubmedArticle's.
    articles_list = []
    if 'PubmedArticleSet' in records and 'PubmedArticle' in records['PubmedArticleSet']:
        articles_list = records['PubmedArticleSet']['PubmedArticle']
        if isinstance(articles_list, dict): # Handle case where only one article is returned as a dict
            articles_list = [articles_list]
    elif 'PubmedArticle' in records: # Sometimes Entrez.read might directly return this if PubmedArticleSet is implicit
        articles_list = records['PubmedArticle']
        if isinstance(articles_list, dict): # Handle single article case
            articles_list = [articles_list]

    for article in articles_list:
        # Accessing keys with correct capitalization as returned by Entrez.read from XML
        pmid_node = article["MedlineCitation"]["PMID"]
        if isinstance(pmid_node, dict):
            pmid = pmid_node.get("text", "")
        else: # it's likely a string
            pmid = str(pmid_node)

        # Safely get the article title
        title_raw = article["MedlineCitation"]["Article"]["ArticleTitle"]
        if isinstance(title_raw, dict): # Sometimes NCBI returns a dict with language tags
            title = title_raw.get("content", "No Title")
        else:
            title = title_raw

        # Safely get the abstract text
        abstract_node = article["MedlineCitation"]["Article"].get("Abstract", {})
        abstract_text_raw = abstract_node.get("AbstractText", [])

        if isinstance(abstract_text_raw, str):
            abstract_text = abstract_text_raw
        elif isinstance(abstract_text_raw, list):
            # AbstractText can be a list of dicts (e.g., {'label': '...', 'content': '...'}) or strings
            abstract_parts = []
            for part in abstract_text_raw:
                if isinstance(part, dict):
                    abstract_parts.append(part.get('content', ''))
                else:
                    abstract_parts.append(str(part))
            abstract_text = " ".join(abstract_parts).strip()
        else:
            abstract_text = ""

        extracted_data.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract_text
        })

    # 6. Save to a JSON file in the Colab environment
    output_filename = "pubmed_abstracts.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(extracted_data, f, indent=2)

    print(f"✅ Success! Saved {len(extracted_data)} abstracts to '{output_filename}'")

    # 7. Preview the first abstract to verify it worked
    if extracted_data: # Only preview if data exists
        print("\n--- PREVIEW OF FIRST ABSTRACT ---")
        print(f"PMID: {extracted_data[0]['pmid']}")
        print(f"Title: {extracted_data[0]['title']}")
        print(f"Abstract: {extracted_data[0]['abstract'][:300]}...") # Prints first 300 characters
    else:
        print("No abstracts were extracted.")
else:
    print("❌ No records found. Try adjusting the search query.")

Searching PubMed for: (single-cell OR spatial transcriptomics) AND (human brain) AND (novel cell type)

Found 10 records. PMIDs: ['34582785', '40585969', '36544231', '41053013', '38350725', '41456076', '40701154', '39217332', '42265312', '41331787']

✅ Success! Saved 10 abstracts to 'pubmed_abstracts.json'

--- PREVIEW OF FIRST ABSTRACT ---
PMID: 34582785
Title: Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.
Abstract: Single-cell gene expression technologies are powerful tools to study cell types in the human brain, but efforts have largely focused on cortical brain regions. We therefore created a single-nucleus RNA-sequencing resource of 70,615 high-quality nuclei to generate a molecular taxonomy of cell types a...
